# PC-RGNN: nuScenes mini object-level detection development track

This is a separate track from the PCL integrity demonstration. It uses the official `v1.0-mini` nuScenes tables and LiDAR files when the user mounts them locally. Raw nuScenes data is intentionally not committed. The model is an object-level development detector built from ground-truth-centered crops; it is not a full-scene proposal system and its mini-split metrics are not official leaderboard results.

## Data contract

Set `NUSCENES_ROOT` to the directory containing the downloaded nuScenes mini release. The notebook uses the official devkit for scenes, categories, poses, boxes, and LiDAR. Train and validation rows are separated by scene token. If the dataset is not mounted, data-dependent cells skip with setup instructions rather than fabricate results.

In [1]:
import os
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NUSCENES_ROOT = Path(os.environ.get('NUSCENES_ROOT', PROJECT_ROOT / 'data' / 'nuscenes'))
VERSION_ROOT = NUSCENES_ROOT / 'v1.0-mini'
DATA_AVAILABLE = (VERSION_ROOT / 'scene.json').exists() or (NUSCENES_ROOT / 'scene.json').exists()
print('NUSCENES_ROOT:', NUSCENES_ROOT)
if not DATA_AVAILABLE:
    print('nuScenes mini not mounted. Download v1.0-mini from the official nuScenes site, then set NUSCENES_ROOT.')

NUSCENES_ROOT: /home/aliaqil/Desktop/projects/pcrgnn/data/nuscenes
nuScenes mini not mounted. Download v1.0-mini from the official nuScenes site, then set NUSCENES_ROOT.


In [2]:
try:
    import torch
    from torch import nn
    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    DEVKIT_AVAILABLE = True
except ImportError as exc:
    DEVKIT_AVAILABLE = False
    print('nuScenes/PyTorch dependencies unavailable:', exc)

if DATA_AVAILABLE and not DEVKIT_AVAILABLE:
    raise ImportError('Install nuscenes-devkit and PyTorch before running the mounted-data track.')
if not DEVKIT_AVAILABLE:
    class _MissingNN:
        Module = object
    nn = _MissingNN()

if DATA_AVAILABLE:
    nusc = NuScenes(version='v1.0-mini', dataroot=str(NUSCENES_ROOT), verbose=False)
    print('loaded official nuScenes version:', nusc.version, '| scenes:', len(nusc.scene))

nuScenes/PyTorch dependencies unavailable: No module named 'nuscenes'


In [3]:
CATEGORY_MAP = {
    'vehicle.car': 'car',
    'vehicle.truck': 'truck',
    'vehicle.bus.bendy': 'bus',
    'vehicle.bus.rigid': 'bus',
    'human.pedestrian.adult': 'pedestrian',
    'human.pedestrian.child': 'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'vehicle.bicycle': 'bicycle',
    'vehicle.motorcycle': 'motorcycle',
}
CLASS_NAMES = sorted(set(CATEGORY_MAP.values()))
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

def split_scene_tokens(scene_records, validation_fraction=0.2):
    tokens = sorted(scene['token'] for scene in scene_records)
    cut = max(1, int(round(len(tokens) * (1 - validation_fraction))))
    return set(tokens[:cut]), set(tokens[cut:])

def crop_from_box(points, center, size, max_points=128, seed=0):
    half = np.asarray(size, dtype=np.float32) * 0.75
    center = np.asarray(center, dtype=np.float32)
    mask = np.all(np.abs(points[:, :3] - center[None, :]) <= half[None, :], axis=1)
    crop = points[mask, :3] - center[None, :]
    if len(crop) == 0:
        return None
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(crop), size=max_points, replace=len(crop) < max_points)
    return crop[indices].astype(np.float32)

def extract_object_crops(nusc, scene_tokens, max_points=128):
    rows = []
    for scene in nusc.scene:
        if scene['token'] not in scene_tokens:
            continue
        sample_token = scene['first_sample_token']
        while sample_token:
            sample = nusc.get('sample', sample_token)
            lidar_token = sample['data']['LIDAR_TOP']
            sample_data = nusc.get('sample_data', lidar_token)
            lidar_path, boxes, _ = nusc.get_sample_data(lidar_token)
            points = LidarPointCloud.from_file(lidar_path).points.T
            for box in boxes:
                class_name = CATEGORY_MAP.get(box.name)
                if class_name is None:
                    continue
                crop = crop_from_box(points, box.center, box.wlh, max_points=max_points, seed=len(rows))
                if crop is None:
                    continue
                yaw = float(box.orientation.yaw_pitch_roll[0])
                target = np.concatenate([np.asarray(box.center, dtype=np.float32), np.asarray(box.wlh, dtype=np.float32), [np.sin(yaw), np.cos(yaw)]])
                rows.append({'scene_token': scene['token'], 'sample_token': sample['token'], 'box_token': box.token, 'class_id': CLASS_TO_ID[class_name], 'class_name': class_name, 'points': crop, 'target': target.astype(np.float32)})
            sample_token = sample['next']
    return rows

if DATA_AVAILABLE:
    train_scenes, val_scenes = split_scene_tokens(nusc.scene)
    train_rows = extract_object_crops(nusc, train_scenes)
    val_rows = extract_object_crops(nusc, val_scenes)
    assert not ({row['scene_token'] for row in train_rows} & {row['scene_token'] for row in val_rows})
    print('train objects:', len(train_rows), '| validation objects:', len(val_rows), '| scene overlap: 0')
else:
    train_rows, val_rows = [], []

In [4]:
class GraphBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Linear(channels, channels, bias=False)
        self.k = nn.Linear(channels, channels, bias=False)
        self.v = nn.Linear(channels, channels, bias=False)
        self.out = nn.Linear(channels, channels)

    def forward(self, x, indices):
        batch, points, channels = x.shape
        expanded = x[:, :, None, :].expand(-1, -1, indices.shape[-1], -1)
        gather = indices[None, :, :, None].expand(batch, -1, -1, channels)
        neighbours = torch.gather(expanded, 1, gather)
        score = (self.q(x)[:, :, None, :] * self.k(neighbours)).sum(-1) / channels**0.5
        weight = score.softmax(-1)
        update = (weight[..., None] * self.v(neighbours)).sum(2)
        return x + self.out(update)

class ObjectPCRGNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = 48
        self.input = nn.Linear(3, channels)
        self.local = GraphBlock(channels)
        self.context = GraphBlock(channels)
        self.class_head = nn.Linear(channels, num_classes)
        self.box_head = nn.Linear(channels, 8)

    def forward(self, points, local_idx, context_idx):
        x = self.input(points)
        x = self.local(x, local_idx)
        x = self.context(x, context_idx)
        pooled = x.mean(1)
        return self.class_head(pooled), self.box_head(pooled)

def fixed_knn(points, k):
    from scipy.spatial import cKDTree
    return cKDTree(points).query(points, k=k + 1)[1][:, 1:]

def batch_from_rows(rows):
    points = np.stack([row['points'] for row in rows]).astype(np.float32)
    labels = np.asarray([row['class_id'] for row in rows], dtype=np.int64)
    targets = np.stack([row['target'] for row in rows]).astype(np.float32)
    local = np.stack([fixed_knn(point, 8) for point in points])
    context = np.stack([fixed_knn(point, 16) for point in points])
    return (torch.tensor(points), torch.tensor(labels), torch.tensor(targets), torch.tensor(local, dtype=torch.long), torch.tensor(context, dtype=torch.long))

def axis_aligned_iou_3d(center_a, size_a, center_b, size_b):
    lower = np.maximum(center_a - size_a / 2, center_b - size_b / 2)
    upper = np.minimum(center_a + size_a / 2, center_b + size_b / 2)
    intersection = np.prod(np.maximum(upper - lower, 0), axis=-1)
    volume_a = np.prod(size_a, axis=-1)
    volume_b = np.prod(size_b, axis=-1)
    return intersection / np.maximum(volume_a + volume_b - intersection, 1e-8)

def summarize_predictions(logits, predictions, labels, targets):
    pred_centers, pred_sizes = predictions[:, :3], np.maximum(predictions[:, 3:6], 1e-3)
    true_centers, true_sizes = targets[:, :3], targets[:, 3:6]
    pred_yaw = np.arctan2(predictions[:, 6], predictions[:, 7])
    true_yaw = np.arctan2(targets[:, 6], targets[:, 7])
    yaw_error = np.abs(np.arctan2(np.sin(pred_yaw - true_yaw), np.cos(pred_yaw - true_yaw)))
    iou = axis_aligned_iou_3d(pred_centers, pred_sizes, true_centers, true_sizes)
    assert np.isfinite(iou).all() and ((iou >= 0) & (iou <= 1)).all()
    assert np.isfinite(yaw_error).all()
    threshold_hits = (iou >= 0.25)
    # This oracle-crop evaluator has one candidate for each GT object; precision and recall are therefore diagnostic hit rates, not full-scene AP.
    hit_rate = float(threshold_hits.mean()) if len(iou) else 0.0
    return {'center_mae_m': float(np.abs(pred_centers - true_centers).mean()), 'size_mae_m': float(np.abs(pred_sizes - true_sizes).mean()), 'yaw_mae_rad': float(yaw_error.mean()), 'axis_aligned_iou3d_mean': float(iou.mean()), 'axis_aligned_iou3d_at_0_25': hit_rate, 'precision_at_0_25': hit_rate, 'recall_at_0_25': hit_rate}

print('model and metric definitions loaded; IoU is explicitly axis-aligned diagnostic IoU, not official oriented nuScenes AP.')

model and metric definitions loaded; IoU is explicitly axis-aligned diagnostic IoU, not official oriented nuScenes AP.


In [5]:
class BatchedGraphBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Linear(channels, channels, bias=False)
        self.k = nn.Linear(channels, channels, bias=False)
        self.v = nn.Linear(channels, channels, bias=False)
        self.out = nn.Linear(channels, channels)

    def forward(self, x, indices):
        batch, points, channels = x.shape
        if indices.ndim == 2:
            indices = indices[None].expand(batch, -1, -1)
        expanded = x[:, :, None, :].expand(-1, -1, indices.shape[-1], -1)
        gather = indices[..., None].expand(-1, -1, -1, channels)
        neighbours = torch.gather(expanded, 1, gather)
        score = (self.q(x)[:, :, None, :] * self.k(neighbours)).sum(-1) / channels**0.5
        weight = score.softmax(-1)
        return x + self.out((weight[..., None] * self.v(neighbours)).sum(2))

class CompletionAwareObjectPCRGNN(nn.Module):
    def __init__(self, num_classes, missing_points=32):
        super().__init__()
        channels = 48
        self.missing_points = missing_points
        self.input = nn.Linear(3, channels)
        self.local = BatchedGraphBlock(channels)
        self.context = BatchedGraphBlock(channels)
        self.completion_head = nn.Sequential(nn.Linear(channels, 64), nn.ReLU(), nn.Linear(64, missing_points * 3))
        self.completed_point_encoder = nn.Sequential(nn.Linear(3, channels), nn.ReLU(), nn.Linear(channels, channels))
        self.feature_head = nn.Sequential(nn.Linear(channels * 2, channels), nn.ReLU())
        self.class_head = nn.Linear(channels, num_classes)
        self.box_head = nn.Linear(channels, 8)

    def forward(self, points, local_idx, context_idx):
        encoded = self.input(points)
        encoded = self.local(encoded, local_idx)
        encoded = self.context(encoded, context_idx)
        observed_feature = encoded.mean(1)
        predicted_missing = self.completion_head(observed_feature).view(-1, self.missing_points, 3)
        completed = torch.cat([points, predicted_missing], dim=1)
        completed_feature = self.completed_point_encoder(completed).mean(1)
        feature = self.feature_head(torch.cat([observed_feature, completed_feature], dim=1))
        return self.class_head(feature), self.box_head(feature), predicted_missing

def completion_batch_from_rows(rows, observed_points=96):
    visible, hidden = [], []
    for index, row in enumerate(rows):
        full = row['points']
        rng = np.random.default_rng(index)
        order = np.argsort(full[:, 0] + rng.normal(0, 0.01, len(full)))
        visible.append(full[order[:observed_points]])
        hidden.append(full[order[observed_points:]])
    points = np.stack(visible).astype(np.float32)
    hidden = np.stack(hidden).astype(np.float32)
    labels = np.asarray([row['class_id'] for row in rows], dtype=np.int64)
    targets = np.stack([row['target'] for row in rows]).astype(np.float32)
    local = np.stack([fixed_knn(point, 8) for point in points])
    context = np.stack([fixed_knn(point, 16) for point in points])
    return (torch.tensor(points), torch.tensor(labels), torch.tensor(targets), torch.tensor(hidden), torch.tensor(local, dtype=torch.long), torch.tensor(context, dtype=torch.long))

def torch_chamfer(predicted, target):
    distances = torch.cdist(predicted, target)
    return distances.min(-1).values.mean() + distances.min(-2).values.mean()

print('completion-aware detector loaded: partial LiDAR -> predicted points -> graph features -> class/box heads')

completion-aware detector loaded: partial LiDAR -> predicted points -> graph features -> class/box heads


In [6]:
if DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows:
    torch.manual_seed(7)
    model = CompletionAwareObjectPCRGNN(len(CLASS_NAMES))
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    train_points, train_labels, train_targets, train_hidden, train_local, train_context = completion_batch_from_rows(train_rows)
    model.train()
    for epoch in range(5):
        optimizer.zero_grad()
        logits, predicted, predicted_missing = model(train_points, train_local, train_context)
        loss = nn.functional.cross_entropy(logits, train_labels) + nn.functional.smooth_l1_loss(predicted, train_targets) + 0.25 * torch_chamfer(predicted_missing, train_hidden)
        loss.backward()
        optimizer.step()
        print(f'epoch {epoch + 1}/5 loss={loss.item():.4f}')
else:
    print('Training skipped: mount nuScenes mini and ensure both scene-disjoint object splits contain rows.')

Training skipped: mount nuScenes mini and ensure both scene-disjoint object splits contain rows.


In [7]:
if DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows:
    model.eval()
    with torch.no_grad():
        val_points, val_labels, val_targets, val_hidden, val_local, val_context = completion_batch_from_rows(val_rows)
        val_logits, val_predictions, val_missing = model(val_points, val_local, val_context)
    metrics = summarize_predictions(val_logits.numpy(), val_predictions.numpy(), val_labels.numpy(), val_targets.numpy())
    predicted_classes = val_logits.argmax(1).numpy()
    print('validation metrics (scene-disjoint nuScenes mini development split):')
    for key, value in metrics.items():
        print(f'{key}: {value:.4f}')
    print('per-class validation counts:')
    for class_id, name in enumerate(CLASS_NAMES):
        mask = val_labels.numpy() == class_id
        class_iou = axis_aligned_iou_3d(val_predictions.numpy()[mask, :3], np.maximum(val_predictions.numpy()[mask, 3:6], 1e-3), val_targets.numpy()[mask, :3], val_targets.numpy()[mask, 3:6]) if mask.any() else np.array([])
        class_hit_rate = float((class_iou >= 0.25).mean()) if len(class_iou) else 0.0
        print(name, 'ground truth:', int(mask.sum()), 'predicted:', int((predicted_classes == class_id).sum()), 'IoU@0.25:', round(class_hit_rate, 4))
else:
    print('Evaluation skipped because nuScenes mini is not available in this environment.')

Evaluation skipped because nuScenes mini is not available in this environment.


## Interpretation

The split is scene-disjoint and every crop retains its scene and box identifiers. The reported center, size, yaw, and IoU quantities are development diagnostics for this compact object-level experiment. The IoU implementation is axis-aligned to keep the notebook auditable; it must not be described as official oriented nuScenes detection AP. Full-scene proposal generation, tracking, calibration analysis, and official benchmark evaluation remain future work.